In [5]:
import pandas as pd
import numpy as np
import os

In [9]:
file_path = "UseCase - Airlines.xlsx"

flights = pd.read_excel(file_path, sheet_name="flights")
payments = pd.read_excel(file_path, sheet_name="payments")
bookings = pd.read_excel(file_path, sheet_name="bookings")
passengers = pd.read_excel(file_path, sheet_name="passengers")

print("Flights:", flights.shape)
print("Payments:", payments.shape)
print("Bookings:", bookings.shape)
print("Passengers:", passengers.shape)

Flights: (1020, 7)
Payments: (1000, 4)
Bookings: (1000, 9)
Passengers: (1039, 9)


In [10]:
print("FLIGHTS")
print(flights.head())
print(flights.isnull().sum())

print("\nPAYMENTS")
print(payments.head())
print(payments.isnull().sum())

print("\nBOOKINGS")
print(bookings.head())
print(bookings.isnull().sum())

print("\nPASSENGERS")
print(passengers.head())
print(passengers.isnull().sum())

FLIGHTS
  flight_id    airline source destination          departure_time  \
0     SJ010   SpiceJet    CCU         MAA 2026-04-20 23:38:41.701   
1     AI155  Air India    BOM         CCU 2026-04-20 23:35:41.703   
2     UK094    Vistara    BOM         CCU 2026-04-20 23:26:41.702   
3     AI245  Air India    BOM         CCU 2026-04-20 23:07:41.704   
4     AI192  Air India    MAA         BOM 2026-04-20 23:05:41.703   

             arrival_time  duration  
0 2026-04-21 02:32:41.701  02:54:00  
1 2026-04-21 01:23:41.703  01:48:00  
2 2026-04-21 01:11:41.702  01:45:00  
3 2026-04-21 01:43:41.704  02:36:00  
4 2026-04-21 04:04:41.703  04:59:00  
flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

PAYMENTS
  payment_id booking_id    amount payment_method
0    PAY1000      B1116   9883.49     NETBANKING
1    PAY1001      B1738   8457.96     NETBANKING
2    PAY1002      B1873   6495.3

In [11]:
flights = flights.drop_duplicates()
payments = payments.drop_duplicates()
bookings = bookings.drop_duplicates()
passengers = passengers.drop_duplicates()

print("Duplicates removed")

Duplicates removed


In [12]:
flights["departure_time"] = pd.to_datetime(
    flights["departure_time"],
    errors="coerce"
)

flights["arrival_time"] = pd.to_datetime(
    flights["arrival_time"],
    errors="coerce"
)

In [13]:
flights["duration"] = flights["duration"].astype(str)

flights["duration"] = pd.to_timedelta(
    flights["duration"],
    errors="coerce"
)

In [14]:
flights["calculated_duration"] = (
    flights["arrival_time"] - flights["departure_time"]
)

In [15]:
flights["duration_minutes"] = (
    flights["calculated_duration"].dt.total_seconds() / 60
)

In [16]:
payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

In [17]:
print(payments["amount"].describe())

count      922.000000
mean      8009.916464
std       4022.055392
min       1002.590000
25%       4623.495000
50%       8027.125000
75%      11288.155000
max      14992.950000
Name: amount, dtype: float64


In [18]:
bookings["booking_date"] = pd.to_datetime(
    bookings["booking_date"],
    errors="coerce"
)

In [20]:
passengers["date_of_birth"] = pd.to_datetime(
    passengers["date_of_birth"],
    errors="coerce"
)

passengers["age"] = pd.to_numeric(
    passengers["age"],
    errors="coerce"
)

In [19]:
passengers_clean = passengers.drop(
    columns=["email", "phone", "aadhaar_id"],
    errors="ignore"
)

bookings_clean = bookings.drop(
    columns=[
        "passport_number",
        "emergency_contact_name",
        "emergency_contact_phone"
    ],
    errors="ignore"
)

In [21]:
analysis_data = bookings_clean.merge(
    flights,
    on="flight_id",
    how="left"
)

In [22]:
analysis_data = analysis_data.merge(
    passengers_clean,
    on="passenger_id",
    how="left"
)

In [23]:
analysis_data = analysis_data.merge(
    payments,
    on="booking_id",
    how="left"
)

In [24]:
print(analysis_data.shape)
print(analysis_data.head())

(1420, 22)
  booking_id passenger_id flight_id            booking_date     status  \
0      B1000        P1591     AI192 2025-06-14 11:37:36.951  CANCELLED   
1      B1001        P1803     6F026 2025-11-02 11:37:36.951  CANCELLED   
2      B1002        P1083     SJ010 2025-08-25 11:37:36.951  CANCELLED   
3      B1002        P1083     SJ010 2025-08-25 11:37:36.951  CANCELLED   
4      B1003        P1364     AI069 2025-12-30 11:37:36.951  CONFIRMED   

  seat_number    airline source destination          departure_time  ...  \
0          3D  Air India    MAA         BOM 2026-04-20 23:05:41.703  ...   
1         18A     IndiGo    BOM         CCU 2026-04-20 22:46:42.000  ...   
2         30C   SpiceJet    CCU         MAA 2026-04-20 23:38:41.701  ...   
3         30C   SpiceJet    CCU         MAA 2026-04-20 23:38:41.701  ...   
4         33A    UNKNOWN    DEL         BOM 2026-04-20 22:43:41.702  ...   

  calculated_duration duration_minutes first_name  last_name age gender  \
0     0 days

In [25]:
print("Missing flight data:")
print(analysis_data["airline"].isnull().sum())

print("Missing passenger data:")
print(analysis_data["first_name"].isnull().sum())

print("Missing payment amount:")
print(analysis_data["amount"].isnull().sum())

Missing flight data:
56
Missing passenger data:
0
Missing payment amount:
454


In [26]:
analysis_data["route"] = (
    analysis_data["source"] + " - " +
    analysis_data["destination"]
)

In [27]:
analysis_data["booking_month"] = (
    analysis_data["booking_date"].dt.to_period("M").astype(str)
)

In [28]:
analysis_data["flight_date"] = (
    analysis_data["departure_time"].dt.date
)

In [29]:
total_flights = flights["flight_id"].nunique()

total_bookings = bookings["booking_id"].nunique()

total_passengers = passengers["passenger_id"].nunique()

total_revenue = payments["amount"].sum()

average_duration = flights["duration_minutes"].mean()

total_airlines = flights["airline"].nunique()

total_routes = flights["route"].nunique() if "route" in flights.columns else (
    flights["source"] + "-" + flights["destination"]
).nunique()

In [30]:
print("Total Flights:", total_flights)
print("Total Bookings:", total_bookings)
print("Total Passengers:", total_passengers)
print("Total Revenue:", total_revenue)
print("Average Duration:", average_duration)
print("Total Airlines:", total_airlines)
print("Total Routes:", total_routes)

Total Flights: 1004
Total Bookings: 1000
Total Passengers: 1000
Total Revenue: 7385142.98
Average Duration: 163.18719810945277
Total Airlines: 5
Total Routes: 30


In [31]:
airline_summary = flights.groupby("airline").agg(
    total_flights=("flight_id", "count"),
    average_duration=("duration_minutes", "mean")
).reset_index()

print(airline_summary)

     airline  total_flights  average_duration
0  Air India            233        163.146051
1     IndiGo            249        167.702951
2   SpiceJet            236        158.089109
3    UNKNOWN             30        161.566667
4    Vistara            218        162.679081


In [32]:
route_summary = flights.groupby(
    ["source", "destination"]
).agg(
    total_flights=("flight_id", "count"),
    average_duration=("duration_minutes", "mean")
).reset_index()

print(route_summary)

   source destination  total_flights  average_duration
0     BLR         BOM             60        147.733582
1     BLR         CCU             21        128.523810
2     BLR         DEL             19        156.947368
3     BLR         HYD             19        155.684211
4     BLR         MAA             16        187.312500
5     BOM         BLR             23        176.522388
6     BOM         CCU             90        169.511221
7     BOM         DEL             39        153.820639
8     BOM         HYD             26        180.846154
9     BOM         MAA             27        183.407407
10    CCU         BLR             20        167.350248
11    CCU         BOM             33        163.969997
12    CCU         DEL             72        153.611180
13    CCU         HYD             21        152.238332
14    CCU         MAA             24        171.291873
15    DEL         BLR             29        170.034826
16    DEL         BOM             28        178.107143
17    DEL 

In [33]:
booking_status = bookings.groupby(
    "status"
).size().reset_index(name="booking_count")

print(booking_status)

      status  booking_count
0  CANCELLED            314
1  CONFIRMED            320
2    INVALID             30
3    PENDING            291


In [34]:
payment_summary = payments.groupby(
    "payment_method"
).agg(
    transactions=("payment_id", "count"),
    total_amount=("amount", "sum")
).reset_index()

print(payment_summary)

  payment_method  transactions  total_amount
0           CARD           329    2400812.04
1     NETBANKING           313    2365644.17
2            UPI           358    2618686.77


In [35]:
os.makedirs("processed", exist_ok=True)

In [36]:
flights.to_csv(
    "processed/clean_flights.csv",
    index=False
)

payments.to_csv(
    "processed/clean_payments.csv",
    index=False
)

bookings_clean.to_csv(
    "processed/clean_bookings.csv",
    index=False
)

passengers_clean.to_csv(
    "processed/clean_passengers.csv",
    index=False
)

analysis_data.to_csv(
    "processed/airline_analysis.csv",
    index=False
)

In [39]:
print("\nFINAL DATASET")
print(analysis_data.shape)

print("\nMissing values:")
print(analysis_data.isnull().sum())

print("\nDuplicate rows:")
print(analysis_data.duplicated().sum())


FINAL DATASET
(1420, 25)

Missing values:
booking_id               0
passenger_id             0
flight_id                0
booking_date             0
status                  59
seat_number              0
airline                 56
source                   0
destination              0
departure_time           0
arrival_time             0
duration                 1
calculated_duration      0
duration_minutes         0
first_name               0
last_name               14
age                      0
gender                   0
date_of_birth            0
payment_id             371
amount                 454
payment_method         371
route                    0
booking_month            0
flight_date              0
dtype: int64

Duplicate rows:
0


In [40]:
print(analysis_data.isnull().sum())

booking_id               0
passenger_id             0
flight_id                0
booking_date             0
status                  59
seat_number              0
airline                 56
source                   0
destination              0
departure_time           0
arrival_time             0
duration                 1
calculated_duration      0
duration_minutes         0
first_name               0
last_name               14
age                      0
gender                   0
date_of_birth            0
payment_id             371
amount                 454
payment_method         371
route                    0
booking_month            0
flight_date              0
dtype: int64


In [43]:
print(flights.columns.tolist())

['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', 'calculated_duration', 'duration_minutes']


In [44]:
print(flights.shape)

(1005, 9)


In [45]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', '_i2', 'categorical_cols', 'col', '_i3', '_i4', '_i5', 'pd', 'np', 'os', '_i6', 'file_path', '_i7', '_i8', '_i9', 'flights', 'payments', 'bookings', 'passengers', '_i10', '_i11', '_i12', '_i13', '_i14', '_i15', '_i16', '_i17', '_i18', '_i19', 'passengers_clean', 'bookings_clean', '_i20', '_i21', 'analysis_data', '_i22', '_i23', '_i24', '_i25', '_i26', '_i27', '_i28', '_i29', 'total_flights', 'total_bookings', 'total_passengers', 'total_revenue', 'average_duration', 'total_airlines', 'total_routes', '_i30', '_i31', 'airline_summary', '_i32', 'route_summary', '_i33', 'booking_status', '_i34', 'payment_summary', '_i35', '_i36', '_i37', '_i38', '_i39', '_i40', '_i41', 'files', '_i42', '_i43', '_i44', '_i45'])


In [46]:
print(analysis_data.shape)
print(analysis_data.columns.tolist())

(1420, 25)
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'seat_number', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', 'calculated_duration', 'duration_minutes', 'first_name', 'last_name', 'age', 'gender', 'date_of_birth', 'payment_id', 'amount', 'payment_method', 'route', 'booking_month', 'flight_date']


In [47]:
analysis_data["status"] = analysis_data["status"].fillna("Unknown")
analysis_data["airline"] = analysis_data["airline"].fillna("Unknown")
analysis_data["last_name"] = analysis_data["last_name"].fillna("Unknown")
analysis_data["payment_id"] = analysis_data["payment_id"].fillna("Unknown")
analysis_data["payment_method"] = analysis_data["payment_method"].fillna("Unknown")

analysis_data["amount"] = analysis_data["amount"].fillna(
    analysis_data["amount"].median()
)

analysis_data["duration"] = analysis_data["duration"].fillna(
    analysis_data["calculated_duration"]
)

In [48]:
print("\nMissing values after filling:")
print(analysis_data.isnull().sum())

print("\nFinal Dataset:", analysis_data.shape)
print("Duplicate rows:", analysis_data.duplicated().sum())


Missing values after filling:
booking_id             0
passenger_id           0
flight_id              0
booking_date           0
status                 0
seat_number            0
airline                0
source                 0
destination            0
departure_time         0
arrival_time           0
duration               0
calculated_duration    0
duration_minutes       0
first_name             0
last_name              0
age                    0
gender                 0
date_of_birth          0
payment_id             0
amount                 0
payment_method         0
route                  0
booking_month          0
flight_date            0
dtype: int64

Final Dataset: (1420, 25)
Duplicate rows: 0


In [49]:
analysis_data.to_csv("final_flights_dataset.csv", index=False)

In [50]:
# Remove decimal values from numeric columns
numeric_columns = analysis_data.select_dtypes(include="number").columns

analysis_data[numeric_columns] = analysis_data[numeric_columns].round(0).astype(int)

# Check
print(analysis_data[numeric_columns].head())

         duration  calculated_duration  duration_minutes  age  amount
0  17940000000000       17940000000000               299    4    8105
1  16020000000000       16020000000000               267   82    8105
2  10440000000000       10440000000000               174   62    6087
3  10440000000000       10440000000000               174   62    2506
4  13560000000000       13560000000000               226   60    8105


In [51]:
analysis_data.to_csv("final_flights_dataset.csv", index=False)

In [52]:
from google.colab import files

files.download("final_flights_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>